# Introduction to Sensor Fusion and Scene Understanding in AD/ADAS
## Python Lab Notebook

**Author:** Federico Camarda  
**Affiliation:** Ampère – Renault Group  

---

This notebook accompanies the lecture *Introduction to Sensor Fusion and Scene Understanding for Autonomous Driving and ADAS*.
It guides you through hands-on implementations of key sensor fusion algorithms in Python.

### Lab Structure

| # | Topic | Status |
|---|-------|--------|
| 1 | Kalman Filter for State Estimation | 🚧 Coming Soon |
| 2 | Multi-Sensor Fusion (Camera · LiDAR · Radar) | 🚧 Coming Soon |
| 3 | Path Prediction & Collision Detection | 🚧 Coming Soon |

> **Note:** This is a preview / mockup version of the notebook. Full exercises will be released alongside the lecture slides.

---
## Setup

Run the cell below to install and import all required libraries.

In [ ]:
# Standard scientific stack — install if needed:
# pip install numpy matplotlib scipy

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import block_diag

print("Libraries loaded successfully.")
print(f"NumPy version: {np.__version__}")

---
## Exercise 1 — Kalman Filter for State Estimation

> 🚧 **Coming Soon** — Full exercise content will be released with the lecture slides.

### Background

The **Kalman Filter** is an optimal recursive estimator for linear systems with Gaussian noise.
It alternates between two steps:

1. **Predict** — propagate the state estimate forward using the motion model  
2. **Update** — correct the prediction using a new measurement

**State vector** (constant-velocity model):
$$\mathbf{x} = [x,\; \dot{x},\; y,\; \dot{y}]^\top$$

**Predict step:**
$$\hat{\mathbf{x}}_{k|k-1} = F \hat{\mathbf{x}}_{k-1|k-1}$$
$$P_{k|k-1} = F P_{k-1|k-1} F^\top + Q$$

**Update step:**
$$K_k = P_{k|k-1} H^\top (H P_{k|k-1} H^\top + R)^{-1}$$
$$\hat{\mathbf{x}}_{k|k} = \hat{\mathbf{x}}_{k|k-1} + K_k (\mathbf{z}_k - H \hat{\mathbf{x}}_{k|k-1})$$
$$P_{k|k} = (I - K_k H) P_{k|k-1}$$

In [ ]:
# ── Kalman Filter skeleton ────────────────────────────────────────────────────
# TODO (Exercise 1): implement the predict and update steps below.

class KalmanFilter:
    """Linear Kalman Filter — constant-velocity 2-D model."""

    def __init__(self, dt=0.1, process_noise=1.0, meas_noise=2.0):
        # State transition matrix  (x, vx, y, vy)
        self.F = np.array([
            [1, dt, 0,  0],
            [0,  1, 0,  0],
            [0,  0, 1, dt],
            [0,  0, 0,  1],
        ])
        # Observation matrix — we only measure position (x, y)
        self.H = np.array([
            [1, 0, 0, 0],
            [0, 0, 1, 0],
        ])
        q = process_noise
        self.Q = block_diag(
            np.array([[dt**3/3, dt**2/2], [dt**2/2, dt]]) * q,
            np.array([[dt**3/3, dt**2/2], [dt**2/2, dt]]) * q,
        )
        self.R = np.eye(2) * meas_noise
        self.x = np.zeros((4, 1))     # state estimate
        self.P = np.eye(4) * 100.0    # estimation covariance

    def predict(self):
        # TODO: implement predict step
        raise NotImplementedError("Coming soon — implement the predict step!")

    def update(self, z):
        # TODO: implement update step
        raise NotImplementedError("Coming soon — implement the update step!")


print("KalmanFilter class defined. Implement predict() and update() to continue.")

In [ ]:
# ── Simulation helper ─────────────────────────────────────────────────────────

def simulate_trajectory(n_steps=60, dt=0.1, seed=42):
    """Generate a noisy ground-truth trajectory with measurements."""
    rng = np.random.default_rng(seed)
    vx, vy = 3.0, 1.5          # true velocity [m/s]
    x, y   = 0.0, 0.0          # initial position
    gt, meas = [], []
    for _ in range(n_steps):
        x += vx * dt + rng.normal(0, 0.05)
        y += vy * dt + rng.normal(0, 0.05)
        gt.append((x, y))
        meas.append((x + rng.normal(0, 1.5),
                      y + rng.normal(0, 1.5)))
    return np.array(gt), np.array(meas)

gt, meas = simulate_trajectory()

plt.figure(figsize=(8, 4))
plt.plot(gt[:, 0],   gt[:, 1],   'g-',  lw=2,  label='Ground truth')
plt.scatter(meas[:, 0], meas[:, 1], s=15, c='orange', alpha=0.6, label='Noisy measurements')
plt.legend(); plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.title('Simulated trajectory — implement the Kalman Filter to denoise the measurements')
plt.tight_layout(); plt.show()

---
## Exercise 2 — Multi-Sensor Fusion

> 🚧 **Coming Soon** — Full exercise content will be released with the lecture slides.

### Background

Real-world autonomous vehicles combine measurements from **multiple sensors**:

| Sensor | Strength | Weakness |
|--------|----------|----------|
| Camera | Rich semantic info, high resolution | No direct depth, sensitive to lighting |
| LiDAR  | Precise 3-D geometry | Sparse at distance, expensive |
| Radar  | Velocity (Doppler), all-weather | Low angular resolution |

A **late-fusion** strategy combines per-sensor object detections into a unified track list.
The placeholder below shows the data structures you will work with.

In [ ]:
# ── Sensor detection data structures ─────────────────────────────────────────

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Detection:
    x: float                       # lateral position [m]
    y: float                       # longitudinal position [m]
    vx: Optional[float] = None     # lateral velocity [m/s]  — radar only
    vy: Optional[float] = None     # longitudinal velocity   — radar only
    sensor: str = 'unknown'        # 'camera' | 'lidar' | 'radar'
    confidence: float = 1.0

# Example detections from three sensors at the same timestep
camera_det = Detection(x=4.1,  y=25.3, sensor='camera', confidence=0.92)
lidar_det  = Detection(x=3.9,  y=25.7, sensor='lidar',  confidence=0.98)
radar_det  = Detection(x=4.5,  y=24.8, vx=-0.2, vy=-8.5, sensor='radar', confidence=0.80)

detections = [camera_det, lidar_det, radar_det]

print("Detections at t=0:")
for d in detections:
    print(f"  [{d.sensor:6s}]  x={d.x:.1f} m  y={d.y:.1f} m  "
          f"vx={d.vx}  vy={d.vy}  conf={d.confidence:.2f}")

print("\nTODO: implement a fusion function that combines these detections into a single estimate.")

---
## Exercise 3 — Path Prediction & Collision Detection

> 🚧 **Coming Soon** — Full exercise content will be released with the lecture slides.

### Background

Once objects are tracked, the system must **predict their future positions** and assess the
risk of collision with the ego vehicle.

A simple **constant-velocity** predictor propagates the current state $N$ steps ahead:
$$\hat{\mathbf{x}}_{k+N} = F^N \hat{\mathbf{x}}_k$$

**Time-to-Collision (TTC)** provides a first-order collision risk estimate:
$$\text{TTC} = -\frac{d}{\dot{d}} \quad \text{if } \dot{d} < 0$$
where $d$ is the current distance and $\dot{d}$ its rate of change.

In [ ]:
# ── TTC skeleton ──────────────────────────────────────────────────────────────

def time_to_collision(ego_pos, ego_vel, obj_pos, obj_vel):
    """
    Estimate Time-to-Collision between ego vehicle and a tracked object.

    Parameters
    ----------
    ego_pos, ego_vel : array-like, shape (2,)  — ego [x, y] position and velocity
    obj_pos, obj_vel : array-like, shape (2,)  — object position and velocity

    Returns
    -------
    float : TTC in seconds, or np.inf if no collision predicted
    """
    # TODO: implement TTC computation
    raise NotImplementedError("Coming soon — implement TTC!")


# Quick sanity check (will raise NotImplementedError until you implement the function)
try:
    ttc = time_to_collision(
        ego_pos=[0, 0], ego_vel=[0, 10],
        obj_pos=[0, 50], obj_vel=[0, -5]
    )
    print(f"TTC = {ttc:.2f} s  (expected ≈ 3.33 s)")
except NotImplementedError as e:
    print(f"Not yet implemented: {e}")

---

## Next Steps

Full solutions and additional exercises will be provided alongside the lecture slides.
In the meantime, feel free to:

- Explore the NumPy / SciPy documentation for linear algebra tools
- Read the classic ["Kalman and Bayesian Filters in Python"](https://github.com/rlabbe/Kalman-and-Bayesian-Filters-in-Python) open textbook
- Look at the [nuScenes dataset](https://www.nuscenes.org/) for real-world multi-sensor data

---
*© Federico Camarda — for educational use only*